## Installing Pre-Requisites

In [ ]:
!pip install datasets
!pip install "chronos[training] @ git+https://github.com/amazon-science/chronos-forecasting.git"
!pip install gluonts
!pip install --upgrade sympy
!pip install transformers
!pip install typer_config
!pip install chronos

  Cloning https://github.com/amazon-science/chronos-forecasting.git to /tmp/pip-install-4n14yqd5/chronos_96543bda7f274b95a1e81616261b1970
  Running command git clone --filter=blob:none --quiet https://github.com/amazon-science/chronos-forecasting.git /tmp/pip-install-4n14yqd5/chronos_96543bda7f274b95a1e81616261b1970
  Resolved https://github.com/amazon-science/chronos-forecasting.git to commit 133761a90145f77971a08c49bfc7cca318b2df9b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Discarding git+https://github.com/amazon-science/chronos-forecasting.git: Requested chronos-forecasting from git+https://github.com/amazon-science/chronos-forecasting.git (from chronos@ git+https://github.com/amazon-science/chronos-forecasting.git) has inconsistent name: expected 'chronos', but metadata has 'chronos-forecasting'
ERROR: Could not find a version that satisfies the requirement chronos (unavailable) (from versi

## Cloning the repository

In [ ]:
import os
from google.colab import drive

# List files in the /content directory
print(os.listdir('/content'))

# Choose a different mountpoint or clear the existing one (if safe)
# For example, to mount to a subdirectory:
mountpoint = '/content/mydrive'
os.makedirs(mountpoint, exist_ok=True) # Create the directory if it doesn't exist

drive.mount(mountpoint)

['.config', 'chronos-forecasting', 'noise-data.arrow', 'mydrive', 'sample_data']
Drive already mounted at /content/mydrive; to attempt to forcibly remount, call drive.mount("/content/mydrive", force_remount=True).


In [ ]:
!git clone https://github.com/themooninhaze/chronos-forecasting.git

fatal: destination path 'chronos-forecasting' already exists and is not an empty directory.


## Loading the Dataset

In [ ]:
from datasets import load_dataset

ds_m4_hourly = load_dataset("autogluon/chronos_datasets", "m4_hourly")

In [ ]:
print(ds_m4_hourly)

DatasetDict({
    train: Dataset({
        features: ['id', 'timestamp', 'target', 'category'],
        num_rows: 414
    })
})


In [ ]:
import pandas as pd
df_m4_hourly = pd.DataFrame(ds_m4_hourly["train"])


In [ ]:
df_m4_hourly.head()

,id,timestamp,target,category
0,T000000,"[2015-01-07 12:00:00, 2015-01-07 13:00:00, 201...","[605.0, 586.0, 586.0, 559.0, 511.0, 443.0, 422...",Other
1,T000001,"[2015-01-07 12:00:00, 2015-01-07 13:00:00, 201...","[3124.0, 2990.0, 2862.0, 2809.0, 2544.0, 2201....",Other
2,T000002,"[2015-01-07 12:00:00, 2015-01-07 13:00:00, 201...","[1828.0, 1806.0, 1897.0, 1750.0, 1679.0, 1620....",Other
3,T000003,"[2015-01-07 12:00:00, 2015-01-07 13:00:00, 201...","[6454.0, 6324.0, 6075.0, 5949.0, 5858.0, 5579....",Other
4,T000004,"[2015-01-07 12:00:00, 2015-01-07 13:00:00, 201...","[4263.0, 4297.0, 4236.0, 4080.0, 3883.0, 3672....",Other


In [ ]:
df_m4_hourly.iloc[0]["timestamp"]

[datetime.datetime(2015, 1, 7, 12, 0),
 datetime.datetime(2015, 1, 7, 13, 0),
 datetime.datetime(2015, 1, 7, 14, 0),
 datetime.datetime(2015, 1, 7, 15, 0),
 datetime.datetime(2015, 1, 7, 16, 0),
 datetime.datetime(2015, 1, 7, 17, 0),
 datetime.datetime(2015, 1, 7, 18, 0),
 datetime.datetime(2015, 1, 7, 19, 0),
 datetime.datetime(2015, 1, 7, 20, 0),
 datetime.datetime(2015, 1, 7, 21, 0),
 datetime.datetime(2015, 1, 7, 22, 0),
 datetime.datetime(2015, 1, 7, 23, 0),
 datetime.datetime(2015, 1, 8, 0, 0),
 datetime.datetime(2015, 1, 8, 1, 0),
 datetime.datetime(2015, 1, 8, 2, 0),
 datetime.datetime(2015, 1, 8, 3, 0),
 datetime.datetime(2015, 1, 8, 4, 0),
 datetime.datetime(2015, 1, 8, 5, 0),
 datetime.datetime(2015, 1, 8, 6, 0),
 datetime.datetime(2015, 1, 8, 7, 0),
 datetime.datetime(2015, 1, 8, 8, 0),
 datetime.datetime(2015, 1, 8, 9, 0),
 datetime.datetime(2015, 1, 8, 10, 0),
 datetime.datetime(2015, 1, 8, 11, 0),
 datetime.datetime(2015, 1, 8, 12, 0),
 datetime.datetime(2015, 1, 8, 13, 

## Converting to Glunout arrow

#### Glunout Function

In [ ]:
from pathlib import Path
from typing import List, Union

import numpy as np
from gluonts.dataset.arrow import ArrowWriter


def convert_to_arrow(
        path: Union[str, Path],
        time_series: Union[List[np.ndarray], np.ndarray],
        compression: str = "lz4",
    ):
        """
        Store a given set of series into Arrow format at the specified path.

        Input data can be either a list of 1D numpy arrays, or a single 2D
        numpy array of shape (num_series, time_length).
        """
        assert isinstance(time_series, list) or (
            isinstance(time_series, np.ndarray) and
            time_series.ndim == 2
        )

        # Set an arbitrary start time
        start = np.datetime64("2000-01-01 00:00", "s")

        dataset = [
            {"start": start, "target": ts} for ts in time_series
        ]

        ArrowWriter(compression=compression).write_to_file(
            dataset,
            path=path,
        )




#### Conversion

In [ ]:
time_series_list = [np.array(target) for target in df_m4_hourly['target']]


In [ ]:

# Convert to GluonTS arrow format
convert_to_arrow("/content/chronos-forecasting/scripts/training/noise-data.arrow", time_series=time_series_list)


## Train.py

In [ ]:
import sys
sys.path.append('/content/chronos-forecasting/scripts/training')
sys.path.append('/content/chronos-forecasting/src/my_chronos')
os.environ['PYTHONPATH'] = "/content/chronos-forecasting/src/my_chronos"
sys.path.append('/content/chronos-forecasting/src/my_chronos')
sys.path.append('/content/chronos-forecasting/scripts/training')

from chronos import ChronosConfig


ImportError: cannot import name 'ChronosConfig' from 'chronos' (/usr/local/lib/python3.10/dist-packages/chronos/__init__.py)

In [ ]:
import sys

if 'chronos' in sys.modules:
    del sys.modules['chronos']

try:
    del chronos
except NameError:
    pass
!pip uninstall -y chronos


Found existing installation: chronos 0.3
Uninstalling chronos-0.3:
  Successfully uninstalled chronos-0.3


In [ ]:
# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: Apache-2.0
import sys
sys.path.append('/content/chronos-forecasting/scripts/training')
sys.path.append('/content/chronos-forecasting/src/chronos')

from pathlib import Path

import pytest
import torch

from chronos import BaseChronosPipeline, ChronosBoltPipeline
from chronos.chronos_bolt import InstanceNorm, Patch
from test.util import validate_tensor


def test_base_chronos_pipeline_loads_from_huggingface():
    BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-tiny", device_map="cpu")


@pytest.mark.parametrize("torch_dtype", [torch.float32, torch.bfloat16])
@pytest.mark.parametrize("input_dtype", [torch.float32, torch.bfloat16, torch.int64])
def test_pipeline_predict(torch_dtype: torch.dtype, input_dtype: torch.dtype):
    pipeline = ChronosBoltPipeline.from_pretrained(
        Path(__file__).parent / "dummy-chronos-bolt-model",
        device_map="cpu",
        torch_dtype=torch_dtype,
    )
    context = 10 * torch.rand(size=(4, 16)) + 10
    context = context.to(dtype=input_dtype)
    expected_num_quantiles = len(pipeline.quantiles)

    # input: tensor of shape (batch_size, context_length)

    quantiles = pipeline.predict(context, prediction_length=3)
    validate_tensor(quantiles, (4, expected_num_quantiles, 3), dtype=torch.float32)

    with pytest.raises(ValueError):
        quantiles = pipeline.predict(
            context, prediction_length=65, limit_prediction_length=True
        )

    quantiles = pipeline.predict(context, prediction_length=65)
    validate_tensor(quantiles, (4, expected_num_quantiles, 65))

    # input: batch_size-long list of tensors of shape (context_length,)

    quantiles = pipeline.predict(list(context), prediction_length=3)
    validate_tensor(quantiles, (4, expected_num_quantiles, 3), dtype=torch.float32)

    with pytest.raises(ValueError):
        quantiles = pipeline.predict(
            list(context),
            prediction_length=65,
            limit_prediction_length=True,
        )

    quantiles = pipeline.predict(list(context), prediction_length=65)
    validate_tensor(quantiles, (4, expected_num_quantiles, 65), dtype=torch.float32)

    # input: tensor of shape (context_length,)

    quantiles = pipeline.predict(context[0, ...], prediction_length=3)
    validate_tensor(quantiles, (1, expected_num_quantiles, 3), dtype=torch.float32)

    with pytest.raises(ValueError):
        quantiles = pipeline.predict(
            context[0, ...],
            prediction_length=65,
            limit_prediction_length=True,
        )

    quantiles = pipeline.predict(
        context[0, ...],
        prediction_length=65,
    )
    validate_tensor(quantiles, (1, expected_num_quantiles, 65), dtype=torch.float32)


@pytest.mark.parametrize("torch_dtype", [torch.float32, torch.bfloat16])
@pytest.mark.parametrize("input_dtype", [torch.float32, torch.bfloat16, torch.int64])
@pytest.mark.parametrize("prediction_length", [3, 65])
@pytest.mark.parametrize(
    "quantile_levels", [[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], [0.1, 0.5, 0.9]]
)
def test_pipeline_predict_quantiles(
    torch_dtype: torch.dtype,
    input_dtype: torch.dtype,
    prediction_length: int,
    quantile_levels: list[int],
):
    pipeline = ChronosBoltPipeline.from_pretrained(
        Path(__file__).parent / "dummy-chronos-bolt-model",
        device_map="cpu",
        torch_dtype=torch_dtype,
    )
    context = 10 * torch.rand(size=(4, 16)) + 10
    context = context.to(dtype=input_dtype)

    num_expected_quantiles = len(quantile_levels)
    # input: tensor of shape (batch_size, context_length)

    quantiles, mean = pipeline.predict_quantiles(
        context,
        prediction_length=prediction_length,
        quantile_levels=quantile_levels,
    )
    validate_tensor(
        quantiles, (4, prediction_length, num_expected_quantiles), dtype=torch.float32
    )
    validate_tensor(mean, (4, prediction_length), dtype=torch.float32)

    # input: batch_size-long list of tensors of shape (context_length,)

    quantiles, mean = pipeline.predict_quantiles(
        list(context),
        prediction_length=prediction_length,
        quantile_levels=quantile_levels,
    )
    validate_tensor(
        quantiles, (4, prediction_length, num_expected_quantiles), dtype=torch.float32
    )
    validate_tensor(mean, (4, prediction_length), dtype=torch.float32)

    # input: tensor of shape (context_length,)

    quantiles, mean = pipeline.predict_quantiles(
        context[0, ...],
        prediction_length=prediction_length,
        quantile_levels=quantile_levels,
    )
    validate_tensor(
        quantiles, (1, prediction_length, num_expected_quantiles), dtype=torch.float32
    )
    validate_tensor(mean, (1, prediction_length), dtype=torch.float32)


# The following tests have been taken from
# https://github.com/autogluon/autogluon/blob/f57beb26cb769c6e0d484a6af2b89eab8aee73a8/timeseries/tests/unittests/models/chronos/pipeline/test_chronos_bolt.py
# Author: Caner Turkmen <atturkm@amazon.com>


def test_given_even_data_patch_operator_output_is_correct():
    batch_size = 17
    patch_len = 16

    patch = Patch(patch_len, patch_len)

    batch = (
        torch.stack([torch.arange(512)] * batch_size)
        + torch.arange(batch_size)[:, None]
    )
    output = patch(batch)

    assert output.shape == (batch_size, 512 // patch_len, patch_len)

    assert torch.allclose(
        output[:, 0],
        torch.stack([torch.arange(patch_len)] * batch_size)
        + torch.arange(batch_size)[:, None],
        atol=1e-5,
    )
    assert torch.allclose(
        output[:, 1],
        torch.stack([torch.arange(patch_len, 2 * patch_len)] * batch_size)
        + torch.arange(batch_size)[:, None],
        atol=1e-5,
    )
    assert not torch.isnan(output).any()


def test_given_even_data_and_strides_patch_operator_output_is_correct():
    batch_size = 17
    patch_len, patch_stride = 16, 8

    patch = Patch(patch_len, patch_stride)

    offset = torch.arange(batch_size)[:, None]
    batch = torch.stack([torch.arange(512)] * batch_size) + offset
    output = patch(batch)

    assert torch.allclose(
        output[:, 1],
        torch.stack([torch.arange(patch_stride, patch_stride + patch_len)] * batch_size)
        + offset,
        atol=1e-5,
    )
    assert not torch.isnan(output).any()


def test_given_uneven_data_patch_operator_pads_and_output_is_correct():
    batch_size = 17
    patch_len = 16

    patch = Patch(patch_len, patch_len)

    batch = (
        torch.stack([torch.arange(512 - patch_len + 1)] * batch_size)
        + torch.arange(batch_size)[:, None]
    ).float()
    output = patch(batch)

    assert output.shape == (batch_size, 512 // patch_len, patch_len)

    # check the first portion is padded
    assert torch.isnan(output[:, 0, :-1]).all()

    # check nowhere else is nan
    assert not torch.isnan(output[:, 1:]).any()


def test_when_instancenorm_applied_then_standardization_correct():
    inorm = InstanceNorm()

    input_ = torch.tensor(
        [
            [1, 2, 3, 4, 5],
            [2, 3, 4, 5, 6],
        ]
    ).float()

    normalized, (loc, scale) = inorm(input_)

    assert normalized.shape == input_.shape
    assert torch.allclose(normalized[0], normalized[1])
    assert torch.allclose(loc.squeeze(), torch.tensor([3.0, 4.0]))
    assert torch.allclose(scale.squeeze(), torch.tensor(1.41421))


def test_when_instancenorm_applied_and_reversed_then_nans_preserved():
    inorm = InstanceNorm()

    input_ = torch.tensor(
        [
            [1, torch.nan, 3, 4, 5],
            [2, 3, 4, 5, torch.nan],
        ]
    ).float()

    normalized, (loc, scale) = inorm(input_)
    assert torch.allclose(normalized.isnan(), input_.isnan())

    output = inorm.inverse(normalized, (loc, scale))
    assert torch.allclose(output, input_, equal_nan=True)


def test_when_instancenorm_applied_and_reversed_then_output_correct():
    inorm = InstanceNorm()

    input_ = torch.tensor(
        [
            [1, 2, 3, 4, 5],
            [2, 3, 4, 5, 1000],
        ]
    ).float()

    normalized, loc_scale = inorm(input_)
    output = inorm.inverse(normalized, loc_scale)

    assert torch.allclose(output, input_)


ModuleNotFoundError: No module named 'chronos.base'

In [ ]:
import sys
sys.path.append('/content/chronos-forecasting/scripts/training')
sys.path.append('/content/chronos-forecasting/src/chronos')


import train

ImportError: attempted relative import with no known parent package